In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix
import plotly.io as pio
import warnings

warnings.filterwarnings("ignore")
pio.renderers.default = "iframe"

# 1. Veri Setlerini İndirme ve İnceleme

In [2]:
df = pd.read_csv(r"/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv",encoding='ISO-8859-1')

In [3]:
df.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [4]:
df.columns = ['target', 'ids', 'date', 'flag', 'user', 'text']

In [5]:
df.head()

,target,ids,date,flag,user,text
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [6]:
df.shape

(1599999, 6)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599999 entries, 0 to 1599998
Data columns (total 6 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   target  1599999 non-null  int64 
 1   ids     1599999 non-null  int64 
 2   date    1599999 non-null  object
 3   flag    1599999 non-null  object
 4   user    1599999 non-null  object
 5   text    1599999 non-null  object
dtypes: int64(2), object(4)
memory usage: 73.2+ MB


# 2. Veri Ön İşleme

In [8]:
df = df[['target', 'text']]

In [9]:
df['target'] = df['target'].replace(4, 1)

In [10]:
df.head()

,target,text
0,0,is upset that he can't update his Facebook by ...
1,0,@Kenichan I dived many times for the ball. Man...
2,0,my whole body feels itchy and like its on fire
3,0,"@nationwideclass no, it's not behaving at all...."
4,0,@Kwesidei not the whole crew


In [11]:
# Gerekli NLTK veri setleri
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [12]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [13]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#','', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return " ".join(words)

In [14]:
df['clean_text'] = df['text'].apply(clean_text)
df.head()

,target,text,clean_text
0,0,is upset that he can't update his Facebook by ...,upset cant update facebook texting might cry r...
1,0,@Kenichan I dived many times for the ball. Man...,dived many time ball managed save 50 rest go b...
2,0,my whole body feels itchy and like its on fire,whole body feel itchy like fire
3,0,"@nationwideclass no, it's not behaving at all....",behaving im mad cant see
4,0,@Kwesidei not the whole crew,whole crew


In [15]:
# Temizleme Sonuçlarını Kıyaslama
ornek = df.sample(1).iloc[0]

print("Eski Hali (Ham)  :", ornek['text'])
print("Yeni Hali (Temiz):", ornek['clean_text'])

Eski Hali (Ham)  : @baxiabhishek  i get what i want, yes. thus stepford-ish behavior you see.  how are you? ham and sausages much? :... http://bit.ly/gjYEc
Yeni Hali (Temiz): get want yes thus stepfordish behavior see ham sausage much


# 3. Metin Duygu Analizi Modeli Oluşturma

In [16]:
df = df.dropna(subset=['clean_text'])

In [17]:
# Veriyi Eğitim ve Test Olarak Ayırma
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], 
    df['target'], 
    test_size=0.2, 
    random_state=42
)

In [18]:
# TF-IDF ile Metinleri Sayısal Formata Çevirme
vectorizer = TfidfVectorizer(max_features=10000) 
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [19]:
# Model Seçimi ve Eğitimi
model = LogisticRegression(max_iter=1000, n_jobs=-1) 
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000, n_jobs=-1)

In [20]:
# Doğruluk Skoru
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

In [21]:
print(f"Metin Duygu Analizi Modeli Doğruluk Skoru: %{accuracy * 100:.2f}")
print("-" * 50)
print("Sınıflandırma Raporu (Classification Report):\n")
print(classification_report(y_test, y_pred))

Metin Duygu Analizi Modeli Doğruluk Skoru: %77.93
--------------------------------------------------
Sınıflandırma Raporu (Classification Report):

              precision    recall  f1-score   support

           0       0.79      0.76      0.77    159494
           1       0.77      0.80      0.78    160506

    accuracy                           0.78    320000
   macro avg       0.78      0.78      0.78    320000
weighted avg       0.78      0.78      0.78    320000



# 4. Metin Duygu Analizi Modeli Optimizasyonu

In [22]:
# Gelişmiş TF-IDF (N-Grams ve Daha Geniş Hafıza)
vectorizer_v2 = TfidfVectorizer(max_features=20000, ngram_range=(1, 2)) 
X_train_tfidf_v2 = vectorizer_v2.fit_transform(X_train)
X_test_tfidf_v2 = vectorizer_v2.transform(X_test)

In [23]:
# Gelişmiş Model (Hiperparametre Ayarı)
model_v2 = LogisticRegression(C=2.0, max_iter=1000, n_jobs=-1) 
model_v2.fit(X_train_tfidf_v2, y_train)

LogisticRegression(C=2.0, max_iter=1000, n_jobs=-1)

In [24]:
y_pred_v2 = model_v2.predict(X_test_tfidf_v2)
accuracy_v2 = accuracy_score(y_test, y_pred_v2)

In [25]:
print(f"Geliştirilmiş Model Doğruluk Skoru: %{accuracy_v2 * 100:.2f}")
print("-" * 50)
print(classification_report(y_test, y_pred_v2))

Geliştirilmiş Model Doğruluk Skoru: %78.92
--------------------------------------------------
              precision    recall  f1-score   support

           0       0.80      0.77      0.78    159494
           1       0.78      0.81      0.79    160506

    accuracy                           0.79    320000
   macro avg       0.79      0.79      0.79    320000
weighted avg       0.79      0.79      0.79    320000



In [26]:
# Eğitilmiş modeli ve TF-IDF dönüştürücüsünü kaydetme
joblib.dump(model_v2, 'nlp_model_v2.pkl')
joblib.dump(vectorizer_v2, 'tfidf_vectorizer_v2.pkl')

print("Modeller başarıyla kaydedildi!")

Modeller başarıyla kaydedildi!


# 5.Model Çıktılarının Görselleştirilmesi

In [27]:
pio.renderers.default = 'iframe'
dist_df = y_test.value_counts().reset_index()
dist_df.columns = ['Duygu', 'Sayı']
dist_df['Duygu'] = dist_df['Duygu'].map({0: 'Negatif', 1: 'Pozitif'})

fig_dist = px.pie(dist_df, values='Sayı', names='Duygu', 
                  title='Test Verisindeki Sınıf Dağılımı (Negatif vs Pozitif)',
                  hole=0.4, color_discrete_sequence=['#EF553B', '#00CC96'])
fig_dist.show(renderer='iframe')

In [28]:
cm = confusion_matrix(y_test, y_pred_v2)
x = ['Tahmin: Negatif', 'Tahmin: Pozitif']
y = ['Gerçek: Negatif', 'Gerçek: Pozitif']

fig_cm = ff.create_annotated_heatmap(z=cm.tolist(), x=x, y=y, 
                                     colorscale='Blues', showscale=True)
fig_cm.update_layout(title_text='NLP Modeli (v2) Karışıklık Matrisi', title_x=0.5)
fig_cm.show(renderer='iframe')

In [29]:
words = vectorizer_v2.get_feature_names_out()
word_weights = np.asarray(X_train_tfidf_v2.sum(axis=0)).flatten()
top_indices = word_weights.argsort()[-20:][::-1]

fig_words = px.bar(x=word_weights[top_indices], y=words[top_indices], orientation='h', 
                   title='Model İçin En Önemli 20 Kelime / Kelime Öbeği (TF-IDF)')
fig_words.update_layout(yaxis={'categoryorder':'total ascending'}, 
                        xaxis_title='Toplam TF-IDF Ağırlığı', yaxis_title='Kelimeler')
fig_words.show(renderer='iframe')

In [30]:
sample_indices = y_test.sample(8, random_state=42).index
sample_texts = X_test.loc[sample_indices]
sample_actuals = y_test.loc[sample_indices].map({0: 'Negatif', 1: 'Pozitif'})

# Pandas serisinin indexlerini hizalayarak tahminleri alalım
preds_series = pd.Series(y_pred_v2, index=y_test.index)
sample_preds = preds_series.loc[sample_indices].map({0: 'Negatif', 1: 'Pozitif'})

results_df = pd.DataFrame({
    'Temizlenmiş Tweet': sample_texts, 
    'Gerçek Duygu': sample_actuals, 
    'Modelin Tahmini': sample_preds
})

fig_table = go.Figure(data=[go.Table(
    columnwidth = [400, 100, 100],
    header=dict(values=list(results_df.columns), fill_color='midnightblue', font=dict(color='white', size=12), align='left'),
    cells=dict(values=[results_df['Temizlenmiş Tweet'], results_df['Gerçek Duygu'], results_df['Modelin Tahmini']], 
               fill_color='aliceblue', align='left'))
])
fig_table.update_layout(title='NLP Modeli Örnek Tahmin Çıktıları')
fig_table.show(renderer='iframe')